# 06 - Prompt Templates and LCEL Q&A Chain

## Purpose

This notebook builds the Q&A chain for Version 2 of the PDF RAG chatbot.

The previous notebook created the vector search layer using OpenAI embeddings, Chroma, and an MMR retriever. This notebook adds the answer-generation layer using GPT and LangChain prompt templates.

The chain retrieves relevant PDF chunks, inserts them into a structured prompt, sends the prompt to GPT, and returns a grounded answer.

## Main Changes in Version 2

This version adds:

- LangChain prompt templates
- GPT-based answer generation
- LCEL chain composition
- Retrieved context passed automatically into the prompt
- Source-aware Q&A behavior

## Main Outputs

- `chat`
- `rag_prompt`
- `chain_retrieving`

## Notebook Flow

```text
User question
        ↓
Retriever searches Chroma
        ↓
Relevant PDF chunks returned
        ↓
Prompt template receives context and question
        ↓
GPT generates answer
        ↓
String output returned


In [0]:
%pip install langchain langchain-community langchain-openai pypdf tiktoken chromadb

In [0]:
dbutils.library.restartPython()

In [0]:
import os

os.environ["OPENAI_API_KEY"] = "OPENAI_API_KEY"

In [0]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

persist_directory = "/tmp/chroma_intro_tableau_v2"

vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding
)

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 2,
        "fetch_k": 6,
        "lambda_mult": 0.7
    }
)

print("Retriever loaded successfully")

In [0]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter, TokenTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
import shutil
import os
import time

pdf_path = "/Volumes/workspace/365pdf/365pdfupload/Introduction_to_Tableau.pdf"

loader_pdf = PyPDFLoader(pdf_path)
docs_list = loader_pdf.load()

markdown_transcript = ""

for doc in docs_list:
    page_number = doc.metadata.get("page", "unknown")
    page_text = doc.page_content

    markdown_transcript += f"\n\n# Introduction to Tableau\n"
    markdown_transcript += f"## Page {page_number}\n\n"
    markdown_transcript += page_text

headers_to_split_on = [
    ("#", "Section Title"),
    ("##", "Page Title")
]

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

docs_list_md_split = md_splitter.split_text(markdown_transcript)

print("Header-split documents:", len(docs_list_md_split))

chat_formatter = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

formatting_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a document-cleaning assistant.

Your task is to clean text extracted from a PDF.

Rules:
- Fix broken line breaks and awkward spacing.
- Improve punctuation and capitalization where needed.
- Keep the original meaning unchanged.
- Do not add new information.
- Do not summarize.
- Do not remove important details.
- Return only the cleaned text.
"""
    ),
    (
        "human",
        """
Clean and format the following extracted PDF text:

{text}
"""
    )
])

formatting_chain = formatting_prompt | chat_formatter | StrOutputParser()

formatted_docs_list = []

for i, doc in enumerate(docs_list_md_split):
    print(f"Formatting document {i + 1} of {len(docs_list_md_split)}")

    text_to_format = doc.page_content[:3000]

    formatted_text = formatting_chain.invoke({
        "text": text_to_format
    })

    formatted_doc = Document(
        page_content=formatted_text,
        metadata=doc.metadata
    )

    formatted_docs_list.append(formatted_doc)

    time.sleep(0.5)

print("Formatted documents:", len(formatted_docs_list))

token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=500,
    chunk_overlap=50
)

docs_list_tokens_split = token_splitter.split_documents(formatted_docs_list)

print("Token-split documents:", len(docs_list_tokens_split))

embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)
from langchain_community.vectorstores import Chroma
import uuid
import os

persist_directory = f"/tmp/chroma_intro_tableau_v2_{uuid.uuid4().hex}"

os.makedirs(persist_directory, exist_ok=True)

print("Using fresh Chroma directory:")
print(persist_directory)

vectorstore = Chroma.from_documents(
    documents=docs_list_tokens_split,
    embedding=embedding,
    persist_directory=persist_directory
)

print("Chroma vector store rebuilt successfully")

try:
    print("Chroma collection count:", vectorstore._collection.count())
except Exception as e:
    print("Could not count collection:", e)

In [0]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 2,
        "fetch_k": 6,
        "lambda_mult": 0.7
    }
)

print("Retriever rebuilt successfully")

In [0]:
test_docs = retriever.invoke("What is Tableau used for?")

print("Retrieved docs:", len(test_docs))

for i, doc in enumerate(test_docs, start=1):
    print(f"\nDocument {i}")
    print("Metadata:", doc.metadata)
    print(doc.page_content[:500])
    print("-" * 80)

In [0]:
def format_docs(docs):
    formatted_text = ""

    for i, doc in enumerate(docs, start=1):
        section_title = doc.metadata.get("Section Title", "Unknown section")
        page_title = doc.metadata.get("Page Title", "Unknown page")

        formatted_text += f"""
Document {i}
Section Title: {section_title}
Page Title: {page_title}

Content:
{doc.page_content}

--------------------
"""

    return formatted_text

In [0]:
formatted_context = format_docs(test_docs)

print(formatted_context[:2000])

In [0]:
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

print("GPT chat model ready")

In [0]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a helpful PDF question-answering assistant.

Answer the user's question using only the retrieved PDF context.

Rules:
- Use only the provided context.
- Do not use outside knowledge.
- If the context does not contain the answer, say:
  "I could not find this information in the uploaded PDF."
- Keep the answer clear and beginner-friendly.
- Do not mention internal implementation details.
"""
    ),
    (
        "human",
        """
Retrieved PDF context:

{context}

User question:

{question}
"""
    )
])

print("RAG prompt template created")

In [0]:
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

chain_retrieving = (
    RunnableParallel(
        {
            "context": RunnableLambda(lambda question: format_docs(retriever.invoke(question))),
            "question": RunnableLambda(lambda question: question)
        }
    )
    | rag_prompt
    | chat
    | StrOutputParser()
)

print("RAG Q&A chain created")

In [0]:
question = "What is Tableau used for?"

answer = chain_retrieving.invoke(question)

print("Question:")
print(question)
print("\nAnswer:")
print(answer)

In [0]:
question = "What are dimensions and measures in Tableau?"

answer = chain_retrieving.invoke(question)

print("Question:")
print(question)
print("\nAnswer:")
print(answer)

In [0]:
question = "What is the capital of Japan?"

answer = chain_retrieving.invoke(question)

print("Question:")
print(question)
print("\nAnswer:")
print(answer)

In [0]:
def ask_pdf_v2(question):
    answer = chain_retrieving.invoke(question)

    print("Question:")
    print(question)
    print("\nAnswer:")
    print(answer)

In [0]:
ask_pdf_v2("How does Tableau help with data visualization?")